# Qwen3-0.6B GraphRAG SFT + LoRA
Run only with human-reviewed train/validation JSONL. No cell uploads artifacts automatically.

In [ ]:
!pip install -q 'transformers>=4.57,<6' 'trl>=0.26,<2' 'peft>=0.18,<1' 'datasets>=4.4,<5' accelerate pyyaml

In [ ]:
from pathlib import Path
import json, os, random
DATASET = Path('/content/reviewed_sft.jsonl')
CONFIG = Path('/content/mxh/chatbot/training/config_sft.yaml')
assert DATASET.exists(), 'Upload reviewed_sft.jsonl first; generated candidates are not approved data.'
rows = [json.loads(line) for line in DATASET.read_text().splitlines() if line.strip()]
assert rows and all(r['split'] in {'train','val'} for r in rows)
assert len({r['sample_id'] for r in rows}) == len(rows)
random.seed(42)
print({'rows': len(rows), 'tasks': sorted({r['task'] for r in rows})})

In [ ]:
import torch
status = {'cuda': torch.cuda.is_available()}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    status.update(device=props.name, vram_gib=round(props.total_memory/1024**3, 2))
print(status)
if not status['cuda']:
    print('TRAIN SKIPPED: use a Colab GPU runtime; local validation can continue without weights.')

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
sample = rows[0]
messages = [{'role':'user','content':sample['prompt']},{'role':'assistant','content':sample['completion']}]
rendered = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
tokens = tokenizer(rendered, truncation=True, max_length=1024)
print({'tokens': len(tokens['input_ids']), 'task': sample['task']})

In [ ]:
# Shared engine: this is the only training implementation used by the notebook and CLI.
RUN_TRAIN = False  # Set True only after dataset review and GPU inspection.
if RUN_TRAIN and torch.cuda.is_available():
    os.environ['ALLOW_MXH_SFT_TRAINING'] = '1'
    !cd /content/mxh/chatbot && python training/train_sft.py --train --dataset /content/reviewed_sft.jsonl --config training/config_sft.yaml
else:
    !cd /content/mxh/chatbot && python training/train_sft.py --dry-run --dataset /content/reviewed_sft.jsonl --config training/config_sft.yaml

In [ ]:
# After training: evaluate, save locally, then run a CPU load smoke only when desired.
ADAPTER = Path('/content/mxh/chatbot/data/models/qwen3-06b-graphrag-lora')
if ADAPTER.exists():
    print('Adapter saved locally:', ADAPTER)
else:
    print('No adapter produced; TRAIN NOT RUN.')